# ETFbook · flujos de ETFs

Los flujos netos por ETF y por mes, con el CSV del repositorio. Incluye el fallo que el plan del proyecto documenta: el script pedia una columna `net_flow` que en los datos se llama `net_flows_usd_m`. La celda 2 lo reproduce y muestra el error real.

In [ ]:
import io, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

URL = ("https://raw.githubusercontent.com/Nicolenki7/ETFbook-data-analytics-case-study/"
       "main/etf_flows_analysis/data/etf_flows_trend_analysis_2025.csv")
with urllib.request.urlopen(URL) as r:
    df = pd.read_csv(io.BytesIO(r.read()))

df["date"] = pd.to_datetime(df["date"])
print("filas:", len(df), "| columnas:", list(df.columns))
print("ETFs:", sorted(df["etf_ticker"].unique()))
print("periodo:", df["date"].min().date(), "->", df["date"].max().date())
df.head()

In [ ]:
# el script original hacia esto:
try:
    df["net_flow"]
except KeyError as e:
    print("KeyError:", e)
    print()
    print("La columna existe, pero se llama distinto. Las columnas reales son:")
    print(list(df.columns))
    print()
    print("=> el arreglo es renombrar en la capa de ingesta, no parchear el SQL.")

In [ ]:
df["mes"] = df["date"].dt.strftime("%Y-%m")
pivote = df.pivot_table(index="mes", columns="etf_ticker",
                        values="net_flows_usd_m", aggfunc="sum")

ax = pivote.plot(kind="bar", figsize=(10, 4))
ax.set_title("Flujos netos por ETF (millones de USD)")
ax.set_ylabel("net_flows_usd_m")
ax.set_xlabel("")
ax.legend(title="ETF", ncols=len(pivote.columns))
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

resumen = (df.groupby("etf_ticker")["net_flows_usd_m"]
             .agg(flujo_total="sum", meses="count")
             .sort_values("flujo_total", ascending=False))
print(f"flujo neto total del periodo: {df['net_flows_usd_m'].sum():,.0f} millones de USD")
print("el ETF con mas flujo concentra el "
      f"{resumen['flujo_total'].iloc[0] / df['net_flows_usd_m'].sum() * 100:.1f}% del total")
resumen